# Tutorial 3: measuring and simulating Hamiltonians (Qiskit)

*Near-term Quantum Computing 2026-2027, tutorial 3.*

In Lecture 3, we wrote Hamiltonians as sums of Pauli strings, estimated their energy from a few
measurement settings, and built Trotter circuits for their time evolution. In this tutorial, we
do the same in code. For a few qubits, we can also compute the exact evolution on a classical
computer, and we use it to check how accurate a Trotter circuit is.

In this tutorial you will learn to:

1. write a Hamiltonian as a sum of Pauli strings in Qiskit, and compute its energies and
   expectation values;
2. build a Trotter circuit from two-qubit Pauli rotations;
3. compute the exact time evolution of a small system, and compare it with the Trotter circuit,
   with and without noise.

**How this tutorial works.** As in Tutorials 1 and 2:

- **Part A** (about 35 minutes): we code together. Type the code yourself in the empty cells.
- **Part B**: exercises, at home, alone or in pairs. Exercises 1 and 2 are the core; 3 and 4 go
  further and rebuild two figures of the lecture.
- Questions marked **(paper)** are for pen and paper: answer them *before* running the code.

**Conventions**, as in Lecture 3: the transverse-field Ising model (TFIM) on an open chain,
$H = -J\sum_i Z_iZ_{i+1} - h\sum_i X_i$, with $J = h = 1$ throughout. The qubits are numbered from
0, as in Qiskit (the lecture notes number them from 1).

Documentation: [Qiskit API reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit),
[`SparsePauliOp`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp),
[`scipy.linalg.expm`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.expm.html).

## Setup

In Google Colab, run the next cell first to install the packages. If you work on your own
computer, install the same packages in your Python environment instead.

In [ ]:
%pip install -q "qiskit~=2.5.2" "qiskit-aer~=0.18.0" pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

simulator = AerSimulator()
J, h = 1.0, 1.0

## Part A: coding together

### A1. Hamiltonians as Pauli sums

In Qiskit, a sum of Pauli strings with coefficients is a
[`SparsePauliOp`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp).
The simplest way to build one is `SparsePauliOp.from_sparse_list`. It takes a list of terms,
each written as `(Pauli string, qubits, coefficient)`, and the number of qubits. For the TFIM on
two qubits, $H = -JZ_0Z_1 - hX_0 - hX_1$, we first write the list of terms, and then build the
operator from it:

In [ ]:
from qiskit.quantum_info import SparsePauliOp

terms = [("ZZ", [0, 1], -J),
         ("X",  [0],    -h),
         ("X",  [1],    -h)]

tfim_2_ham = SparsePauliOp.from_sparse_list(terms, num_qubits=2)
print(tfim_2_ham)

Qiskit prints each term as a label with qubit 0 on the right, as in the bitstrings of Tutorial 1:
`IX` is $X_0$ and `XI` is $X_1$. With `from_sparse_list`, we never have to write these labels
ourselves.

`tfim_2_ham.to_matrix()` gives the $4\times4$ matrix of $H$, and `np.linalg.eigvalsh` the eigenvalues of
a Hermitian matrix, in increasing order. In the notes exercise "The TFIM on two qubits", the
ground-state energy is $-\sqrt{J^2 + 4h^2}$. Check it:

In [ ]:
# energies: the eigenvalues of tfim_2_ham. Print them, and the formula for the lowest one.
# YOUR CODE HERE

For a chain of $n$ qubits, we build the same list of terms with two loops: the $n-1$ bonds and the
$n$ fields. (`terms.append(x)` adds `x` at the end of the list `terms`.)

In [ ]:
def tfim(n, J, h):
    """The TFIM on an open chain of n qubits, as a SparsePauliOp."""
    # YOUR CODE HERE


print(tfim(4, J, h))
print("number of terms for n = 10:", len(tfim(10, J, h)))

**Expectation values.** As in Tutorial 1, `state.expectation_value(H)` computes
$\langle\psi|H|\psi\rangle$ for a `Statevector`, now for a whole sum of Pauli strings. The result
is a complex number with zero imaginary part; `.real` keeps its real part.
`Statevector.from_label("0000")` is the state $|0000\rangle$, and `Statevector.from_label("++++")`
is $|{+}{+}{+}{+}\rangle$.

**(paper)** What is $\langle H\rangle$ for $n = 4$ in these two states? Then check with the code.

In [ ]:
tfim_4_ham = tfim(4, J, h)
# Print <H> in |0000> and in |++++>
# YOUR CODE HERE

*Your answer:*

Any other observable that is a sum of Pauli strings is built in the same way. The magnetisation
$m = \frac1n\sum_i\langle Z_i\rangle$ of the lecture is the expectation value of this operator:

In [ ]:
def magnetisation(n):
    """The operator (Z_0 + ... + Z_{n-1}) / n, whose expectation value is the magnetisation."""
    terms = []
    for i in range(n):
        terms.append(("Z", [i], 1 / n))
    return SparsePauliOp.from_sparse_list(terms, num_qubits=n)


print("m in |0000>:", Statevector.from_label("0000").expectation_value(magnetisation(4)).real)

### A2. A Trotter step

A Trotter step of the TFIM applies $e^{-iA\delta}$, with $A$ the bond terms, and then
$e^{-iB\delta}$, with $B$ the field terms. Each is a product of Pauli rotations, one per term.
Qiskit has a gate for the rotation about each two-qubit Pauli string, with the same convention as
`rx`:

| gate | `rx(θ, q)` | `rzz(θ, i, j)` | `rxx(θ, i, j)` | `ryy(θ, i, j)` |
| --- | --- | --- | --- | --- |
| unitary | $e^{-i\theta X_q/2}$ | $e^{-i\theta Z_iZ_j/2}$ | $e^{-i\theta X_iX_j/2}$ | $e^{-i\theta Y_iY_j/2}$ |

So a bond term gives $e^{iJ\delta Z_iZ_{i+1}}$ = `rzz(-2 * J * dt, i, i + 1)`, and a field term
$e^{ih\delta X_i}$ = `rx(-2 * h * dt, i)`. As in the lecture, the bonds go in two layers, first
$(0,1), (2,3), \dots$, then $(1,2), (3,4), \dots$, and the fields in a third.
`range(0, n - 1, 2)` counts $0, 2, 4, \dots$ up to $n-2$.

In [ ]:
def trotter_step(n, J, h, dt):
    """One Trotter step of length dt for the TFIM on n qubits."""
    step = QuantumCircuit(n)
    # YOUR CODE HERE
    return step


print(trotter_step(6, J, h, 0.25).draw())

To evolve for a time $t$, we repeat the step `steps` times with $\delta = t/\text{steps}$.
`a.compose(b, inplace=True)` appends circuit `b` to the end of `a` (Tutorial 2, Exercise 1).
`inplace=True` changes `a` itself; without it, `compose` leaves `a` unchanged and returns a new,
combined circuit.

In [ ]:
def trotter_circuit(n, J, h, t, steps):
    """`steps` Trotter steps of the TFIM up to time t, starting from |0...0>."""
    circuit = QuantumCircuit(n)
    # YOUR CODE HERE
    return circuit

**(paper)** How many CZs does one step on 6 qubits need? Check by compiling the step to the native
gates `r` and `cz`, as in Tutorial 2, now with `optimization_level=1`: the compiler then also
merges neighbouring single-qubit gates. (Try `optimization_level=0` too, and compare the counts.)

In [ ]:
# Compile trotter_step(6, J, h, 0.25) to r and cz (optimization_level=1) and count its gates
# YOUR CODE HERE

*Your answer:*

### A3. Exact evolution, and how good the circuit is

For a few qubits, we can compute $e^{-iHt}$ exactly as a matrix. `scipy.linalg.expm` computes the
exponential of a matrix (not `np.exp`, which takes the exponential of each entry separately). The
matrix has $2^n\times2^n$ entries, so this works up to about 12 qubits.
`state.evolve(U)` applies a matrix `U` to a `Statevector` and returns the new state.

In [ ]:
from scipy.linalg import expm


def exact_state(n, J, h, t):
    """The exact state e^{-iHt}|0...0> of the TFIM on n qubits."""
    # YOUR CODE HERE

Now we compare the Trotter circuit with the exact state, for 6 qubits at $t = 1$, with more and
more steps. We compare two things: the magnetisation, and the fidelity of Lecture 2, which for
two pure states is $|\langle\phi|\psi\rangle|^2$ and is computed by `state_fidelity`. (`round(x, 4)` rounds `x` to 4 decimals, to keep the printout short.)

In [ ]:
from qiskit.quantum_info import state_fidelity

n, t = 6, 1.0
exact = exact_state(n, J, h, t)
M = magnetisation(n)
print("exact: m =", round(exact.expectation_value(M).real, 4))

# For 1, 2, 4, 8 and 16 steps: print the magnetisation of the Trotter state and its fidelity with `exact`
# YOUR CODE HERE

**Think.** By which factor does the infidelity $1 - F$ shrink each time the number of steps
doubles? Compare with the Trotter error bound of Lecture 3.

*Your answer:*

### Summary: from Lecture 3 to code

| Lecture 3 | Qiskit |
| --- | --- |
| $H = \sum_i c_iP_i$ | `terms = [("ZZ", [0, 1], c), ...]`, then `SparsePauliOp.from_sparse_list(terms, num_qubits=n)` |
| the matrix of $H$, its eigenvalues | `H.to_matrix()`, `np.linalg.eigvalsh(H.to_matrix())` |
| $\langle\psi\vert H\vert\psi\rangle$ | `state.expectation_value(H).real` |
| $e^{-i\theta X_iX_j/2}$, $e^{-i\theta Y_iY_j/2}$, $e^{-i\theta Z_iZ_j/2}$ | `rxx(θ, i, j)`, `ryy(θ, i, j)`, `rzz(θ, i, j)` |
| $e^{-iHt}$, exactly | `expm(-1j * t * H.to_matrix())` |
| $U\vert\psi\rangle$ | `state.evolve(U)` |
| fidelity of two states | `state_fidelity(a, b)` |

## Part B: exercises

Exercises 1 and 2 are the core of this tutorial; 3 and 4 go further. Use the functions from Part A.

### Exercise 1 (core): the energy from two settings

In Lecture 3, we measured the TFIM in two settings. In the $Z$ setting, each shot gives the signs
$z_i = \pm1$ of all qubits, and so one value of the bond group, $s_Z = -J\sum_i z_iz_{i+1}$. In the
$X$ setting (a Hadamard on every qubit before the measurement), each shot gives signs $x_i$, and
one value of the field group, $s_X = -h\sum_i x_i$.

The mean of the values of a group estimates its expectation value, and the standard error of
the mean is $\sqrt{\mathrm{Var}(s_g)/N}$, with $\mathrm{Var}(s_g)$ estimated by the sample
variance of the $N$ values. This variance includes the covariances between the terms of the
group (notes, section "Measurement settings"). The two settings use different shots, so
$\langle H\rangle$ is the sum of the two means, and its squared standard error is the sum of the two
squared standard errors, $\mathrm{Var}(s_Z)/N + \mathrm{Var}(s_X)/N$.
This replaces `estimate_z_product` of Tutorials 1 and 2.

The function `signs` turns a bitstring into a list of signs, one per qubit:

In [ ]:
def signs(bitstring):
    """The outcomes of one shot as signs: signs(bitstring)[q] is +1 or -1 for qubit q."""
    return [1 - 2 * int(b) for b in bitstring[::-1]]   # reversed: Qiskit writes qubit 0 last


print(signs("0001"))   # qubit 0 gave 1

**(a) (code)** Complete `group_values`, which returns one value of the group for each shot. A
bitstring that appears `count` times gives `count` equal values; `values += [value] * count` adds
them all to the list. For the $Z$ setting, `s[i] * s[i + 1]` is $z_iz_{i+1}$: add these up
with a loop over the bonds. For the $X$ setting, `sum(s)` adds up all the signs.

In [ ]:
def group_values(counts, setting, J, h):
    """One value per shot: s_Z if setting is "Z", s_X if setting is "X"."""
    values = []
    for bitstring, count in counts.items():
        s = signs(bitstring)
        n = len(s)
        # YOUR CODE HERE
        values += [value] * count
    return np.array(values)

**(b) (code)** Complete `measure_energy`. For each setting, copy the circuit that prepares the
state, add the Hadamards for the $X$ setting (`circuit.h(range(n))` applies one to every qubit),
measure, and use `group_values`. For a numpy array `values`, `values.mean()` and `values.var()`
are the sample mean and variance. The last cell then estimates $\langle H\rangle$ of the state after $t = 1$
(6 qubits, 10 steps) with 100, 1000 and 10 000 shots per setting: compare with the exact
expectation value in the state of the circuit.

In [ ]:
def measure_energy(state_circuit, shots, J, h):
    """Estimate <H> of the TFIM from `shots` shots in each of the two settings.

    Returns the estimate and its standard error."""
    n = state_circuit.num_qubits
    energy, variance = 0, 0
    for setting in ["Z", "X"]:
        circuit = state_circuit.copy()
        # Add the Hadamards if setting is "X", measure all qubits, run `shots` shots, and use
        # group_values. Then add the mean of the values to `energy`, and the squared standard
        # error values.var() / shots to `variance`.
        # YOUR CODE HERE
    return energy, np.sqrt(variance)


n = 6
state_circuit = trotter_circuit(n, J, h, t=1.0, steps=10)
exact_energy = Statevector(state_circuit).expectation_value(tfim(n, J, h)).real
print("exact <H> =", round(exact_energy, 3))

for shots in [100, 1000, 10_000]:
    energy, error = measure_energy(state_circuit, shots, J, h)
    print(shots, "shots per setting: <H> =", round(energy, 3), "±", round(error, 3))

**(c) (paper)** From your result with 10 000 shots per setting, how many shots per setting would a
standard error of 0.01 need? Why can we not use the formula $\sqrt{(1 - \langle P\rangle^2)/N}$
of Tutorial 1 for a group?

*Your answer:*

### Exercise 2 (core): how many Trotter steps?

More Trotter steps make the Trotter error smaller, but every step adds gates, and with them noise.
In Lecture 3, a simple model of the two errors gave a best number of steps $r^*$; here we find it
by simulation, as for the figure of the notes. We use the depolarising noise model and the
density-matrix simulation of Tutorial 2 (copied below), with the values of today's devices quoted
there: $p = 0.1\%$ after every `r` gate and $0.5\%$ after every CZ. (Tutorial 2 used values five
times larger.) We compile with `optimization_level=1`, as in A2, so the fidelities are slightly
higher than in the figure of the notes, which compiles without merging gates. `state_fidelity` also accepts a density matrix $\rho$, and then computes
$\langle\psi|\rho|\psi\rangle$.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error


def native_noise_model(p1, p2):
    """Depolarising errors after every r gate (p1) and every cz gate (p2), as in Tutorial 2, A2."""
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["r"])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cz"])
    return noise_model


def noisy_density_matrix(circuit, noise_model):
    """Exact final density matrix of `circuit` (native gates, no measurements) under `noise_model`.

    From Tutorial 2, A3."""
    circuit = circuit.copy()
    circuit.save_density_matrix()
    density_simulator = AerSimulator(method="density_matrix", noise_model=noise_model)
    return density_simulator.run(circuit).result().data()["density_matrix"]

**(a) (code)** For 4 qubits and $t = 1$, compute the fidelity with the exact state for
$r = 1, \dots, 12$ steps, without noise and with `noise_model`. As in Tutorial 2, compile each
circuit to `r` and `cz` (with `optimization_level=1`, as in A2) before the noisy simulation. Which number of
steps is best?

The plotting code is given. `plt.plot(x, y, "o-", label=...)` draws the points `(x, y)` as circles
(`"o"`) joined by lines (`"-"`); `"s"` draws squares instead. `plt.legend()` shows the labels.
`np.argmax(list)` gives the position of the largest entry.

In [ ]:
n, t = 4, 1.0
step_numbers = range(1, 13)
exact = exact_state(n, J, h, t)
noise_model = native_noise_model(0.001, 0.005)

# fidelity_ideal, fidelity_noisy: lists with one fidelity per number of steps
# YOUR CODE HERE

plt.plot(step_numbers, fidelity_ideal, "o-", label="no noise")
plt.plot(step_numbers, fidelity_noisy, "s-", label="with gate noise")
plt.xlabel("Trotter steps r")
plt.ylabel("fidelity with the exact state")
plt.legend()
plt.show()
print("best number of steps with noise:", step_numbers[np.argmax(fidelity_noisy)])

**(b) (paper, then code)** In the lecture, the model $1 - F \approx (at^2/r)^2 + r\,\varepsilon_{\text{step}}$
gave the best number of steps $r^* = (2a^2t^4/\varepsilon_{\text{step}})^{1/3}$, where
$\varepsilon_{\text{step}}$ is the error probability of one step. With the noise of Tutorial 2,
$p = 0.5\%$ and $3\%$, both error probabilities are 5 to 6 times larger, and so is
$\varepsilon_{\text{step}}$, roughly the sum of the error probabilities of the gates of a step
(about 5.5 times). By which factor should
$r^*$ change? Check by adding the curve with `native_noise_model(0.005, 0.03)` to the plot.

In [ ]:
# fidelity_t2: the fidelity with the noise of Tutorial 2, one value per number of steps
# YOUR CODE HERE

plt.plot(step_numbers, fidelity_ideal, "o-", label="no noise")
plt.plot(step_numbers, fidelity_noisy, "s-", label="p = 0.1%, 0.5%")
plt.plot(step_numbers, fidelity_t2, "^-", label="p = 0.5%, 3% (Tutorial 2)")
plt.xlabel("Trotter steps r")
plt.ylabel("fidelity with the exact state")
plt.legend()
plt.show()
print("best number of steps with the noise of Tutorial 2:", step_numbers[np.argmax(fidelity_t2)])

*Your answer:*

### Exercise 3 (further): small and large steps

Here we rebuild the figure of Lecture 3 with Trotter steps of 0.25, 0.5 and 1: the magnetisation
and the fidelity with the exact state over time, for 6 qubits, after each full step.

`evolve` also takes a circuit: `state.evolve(circuit)` applies the circuit to `state` and returns
the new state (it does not change `state` itself, so write `state = state.evolve(circuit)`). So we
can apply one step at a time, and look at the state after every step, without building a new
circuit for each time.

**(a) (code)** Complete `trotter_states`, which applies one step at a time with `evolve`, and
returns the list of states after $0, 1, \dots$ steps. Then complete the loop below.

The plotting code is given. `plt.subplots(1, 2)` makes a figure with two panels side by side,
`ax_m` (magnetisation) and `ax_f` (fidelity); `ax_m.plot(...)` draws in the left panel, as
`plt.plot` did in Exercise 2. The exact curve is computed at 81 times between 0 and 4
(`np.linspace(0, 4, 81)`). For each step size, `zip` goes through the step sizes, colours and
markers together, and `marker + "--"` joins the points with dashed lines.

In [ ]:
def trotter_states(n, J, h, dt, n_steps):
    """The states after 0, 1, ..., n_steps Trotter steps of length dt, starting from |0...0>."""
    # YOUR CODE HERE

In [ ]:
n, t_max = 6, 4.0
M = magnetisation(n)

fine_times = np.linspace(0, t_max, 81)
exact_m = [exact_state(n, J, h, t).expectation_value(M).real for t in fine_times]

fig, (ax_m, ax_f) = plt.subplots(1, 2, figsize=(11, 4))
ax_m.plot(fine_times, exact_m, color="tab:orange", lw=2, label="exact")
blues = ["#6baed6", "#2171b5", "#08306b"]           # light to dark: small to large steps
for dt, colour, marker in zip([0.25, 0.5, 1.0], blues, ["o", "s", "^"]):
    n_steps = int(t_max / dt)
    times = dt * np.arange(n_steps + 1)              # 0, dt, 2 dt, ..., t_max
    # m, fidelity: lists with the magnetisation and the fidelity with the exact state, one per time.
    # For k in range(n_steps + 1), use states[k] and exact_state(n, J, h, times[k]).
    # YOUR CODE HERE
    ax_m.plot(times, m, marker + "--", color=colour, label="step " + str(dt))
    ax_f.plot(times, fidelity, marker + "--", color=colour, label="step " + str(dt))

ax_m.set_xlabel("time t")
ax_m.set_ylabel("magnetisation m")
ax_m.legend()
ax_f.set_xlabel("time t")
ax_f.set_ylabel("fidelity with the exact state")
plt.show()

**(b) (paper)** For which step size does the magnetisation follow the exact curve? Compare the
two panels at step 0.5: why can the magnetisation be close to the exact value while the fidelity
is low? What does the circuit with step 1 simulate?

*Your answer:*

### Exercise 4 (further): the light cone

In Lecture 3, we saw how a flipped qubit spreads through a Heisenberg chain,
$H = J\sum_i (X_iX_{i+1} + Y_iY_{i+1} + Z_iZ_{i+1})$. We start from $|0\cdots0\rangle$ with the
middle qubit flipped. The model conserves the number of flipped qubits, so there is always exactly
one, and $(1 - \langle Z_i\rangle)/2$ is the probability of finding it on qubit $i$.

On one bond, $X_iX_{i+1}$, $Y_iY_{i+1}$ and $Z_iZ_{i+1}$ commute, so $e^{-iJ\delta(XX + YY + ZZ)}$
on a bond is exactly the product of three rotations. A Trotter step applies these three gates to
the bonds $(0,1), (2,3), \dots$, then to $(1,2), (3,4), \dots$.

**(a) (code)** Complete `heisenberg_step`, using the table of A2 for the angles.

In [ ]:
def heisenberg_step(n, J, dt):
    """One Trotter step of length dt for the Heisenberg chain on n qubits."""
    step = QuantumCircuit(n)
    # YOUR CODE HERE
    return step


print(heisenberg_step(4, J, 0.2).draw())

**(b) (code)** For 15 qubits, flip the middle qubit (qubit 7), apply 30 steps of length 0.2 with
`evolve`, and before the first step and after every step store the probability of finding the flip on
each qubit. $Z_i$ is `SparsePauliOp.from_sparse_list([("Z", [i], 1)], num_qubits=n)`, and the
start state is the `Statevector` of a circuit with an `x` gate on the middle qubit. The plotting
code is given. It draws the probabilities as a coloured grid, with the time from left to right and
qubit 0 at the top, as in a circuit drawing:

- `np.array(flip_probability).T` turns your list into a table with one row per qubit and one
  column per time (`.T` transposes it);
- `plt.pcolormesh(x, y, table)` colours each cell of the table, at the times `x` and the qubits
  `y`, and `plt.colorbar` shows which colour is which value;
- `norm=LogNorm(vmin=1e-3, vmax=1)` makes the colour scale logarithmic, from $10^{-3}$ to 1, so
  that small probabilities are visible too; `np.maximum(..., 1e-3)` draws values below $10^{-3}$
  (and the zeros, whose logarithm does not exist) with the colour of $10^{-3}$;
- the white dashed lines are the light cone of the circuit, 2 qubits per side per step;
  `plt.ylim(n - 0.5, -0.5)` puts qubit 0 at the top.

In [ ]:
n, dt, n_steps = 15, 0.2, 30
middle = n // 2

# flip_probability: a list with one entry per time (0, 1, ..., n_steps steps);
# each entry is a list of the n probabilities (1 - <Z_i>) / 2
# YOUR CODE HERE

from matplotlib.colors import LogNorm   # a logarithmic colour scale

times = dt * np.arange(n_steps + 1)
probabilities = np.maximum(np.array(flip_probability).T, 1e-3)   # rows: qubits, columns: times
plt.figure(figsize=(12, 3.5))
plt.pcolormesh(times, range(n), probabilities, norm=LogNorm(vmin=1e-3, vmax=1), shading="nearest")
plt.colorbar(label="probability of the flip on qubit i")
k = np.arange(n_steps + 1)
plt.plot(times, middle + 2 * k, "w--")
plt.plot(times, middle - 2 * k, "w--")
plt.ylim(n - 0.5, -0.5)
plt.xlabel("time t")
plt.ylabel("qubit i")
plt.show()

**(c) (paper)** Outside the dashed lines, the probability is exactly zero, up to rounding errors
(the cell below checks this). Why? The coloured region
spreads more slowly than the dashed lines: what sets its speed? What happens once it reaches the ends
of the chain?

In [ ]:
# outside: the probabilities at qubits more than 2k qubits from the middle after k steps
outside = []
for k in range(n_steps + 1):
    for i in range(n):
        if abs(i - middle) > 2 * k:
            outside.append(flip_probability[k][i])
print("largest probability outside the circuit cone:", max(outside))

*Your answer:*

## Wrap-up

- A Hamiltonian is a `SparsePauliOp`: its matrix gives exact energies and evolutions for small
  systems, and its expectation values can be computed exactly or estimated from shots.
- With grouped settings, each shot gives one value of a whole group; the sample variance of
  these values gives the standard error, covariances included.
- A Trotter step is a circuit of Pauli rotations; its error falls with the number of steps, while
  the gate noise grows with it, and simulating small instances finds the best compromise.
- Local observables can be accurate long before the whole state is.

Next week, we look for ground states: the energy estimate of Exercise 1 becomes the cost function
of an optimisation.